# 1. B2B CUSTOMER SEGMENTATION (K-MEANS CLUSTERING)
**Objective:** Extract operational B2B sales data via SQL Server, engineer RFM features, and apply K-Means to detect behavioral churn risks. Includes dynamic ML persona mapping to prevent label scrambling.

In [33]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import pyodbc 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')
import datetime as dt

print("Libraries successfully imported.")

Libraries successfully imported.


### 1.1. Establishing Database Connection & Advanced Extraction
Connecting to RheinTrade SQL Server. The query explicitly integrates SCD Type 2 (Historical Costing) from `ProductPriceHistory` and deduces returns to align perfectly with Power BI DAX measures (`Total Revenue` and `Total Profit`).

In [34]:
# 2. ESTABLISH DATABASE CONNECTION & EXTRACT DATA
server = 'DESKTOP-O8PCC7J\\SQLEXPRESS03' 
database = 'RheinTrade Solutions'
conn_str = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    rf'SERVER={server};'
    rf'DATABASE={database};'
    r'Trusted_Connection=yes;'
)

try:
    conn = pyodbc.connect(conn_str)
    
    query = """
    WITH BaseOrderData AS (
        SELECT 
            c.CustomerID, c.CompanyName, c.CustomerType, c.IsActive, cs.SegmentName AS CurrentSegment,
            o.OrderID, o.OrderDate, od.OrderDetailID, od.ProductID, od.Quantity, od.UnitPrice, od.DiscountRate,
            COALESCE(pph.CostPrice, p.CostPrice) AS HistoricalUnitCost
        FROM Orders o
        JOIN OrderDetails od ON o.OrderID = od.OrderID
        JOIN Customers c ON o.CustomerID = c.CustomerID
        LEFT JOIN CustomerSegments cs ON c.SegmentID = cs.SegmentID 
        JOIN Products p ON od.ProductID = p.ProductID
        
        LEFT JOIN ProductPriceHistory pph 
            ON od.ProductID = pph.ProductID 
            AND o.OrderDate >= pph.StartDate 
            AND o.OrderDate <= pph.EndDate
            
        WHERE o.OrderStatus IN ('Delivered', 'Partially Returned', 'Returned')
    ),
    AggregatedReturns AS (
        SELECT 
            OrderDetailID, 
            SUM(QuantityReturned) AS TotalQuantityReturned,
            SUM(RefundAmount) AS TotalRefundAmount
        FROM Returns
        GROUP BY OrderDetailID
    )
    SELECT 
        b.CustomerID, b.CompanyName, b.CustomerType, b.IsActive, b.CurrentSegment, 
        b.OrderID, b.OrderDate,
        
        ((b.Quantity * b.UnitPrice * (1 - b.DiscountRate)) - ISNULL(r.TotalRefundAmount, 0)) AS NetRevenue,
        
        (((b.Quantity * b.UnitPrice * (1 - b.DiscountRate)) - ISNULL(r.TotalRefundAmount, 0)) - 
         ((b.Quantity * b.HistoricalUnitCost) - (ISNULL(r.TotalQuantityReturned, 0) * b.HistoricalUnitCost))) AS NetProfit
         
    FROM BaseOrderData b
    LEFT JOIN AggregatedReturns r ON b.OrderDetailID = r.OrderDetailID
    """
    
    df_raw = pd.read_sql(query, conn)
    print(f"Advanced data extraction complete. Total rows retrieved: {df_raw.shape[0]}")
    
except pyodbc.Error as e:
    print("Database connection failed:", e)
finally:
    if 'conn' in locals():
        conn.close()

Advanced data extraction complete. Total rows retrieved: 1735


### 2. Feature Engineering: RFM Metrics
We filter for Active B2B Customers and aggregate the transaction data into Recency, Frequency, Monetary (Revenue), and Net Profit features.

In [35]:
# 3. FEATURE ENGINEERING: RFM METRICS FOR ACTIVE B2B CUSTOMERS
df_b2b = df_raw[(df_raw['CustomerType'] == 'B2B') & (df_raw['IsActive'] == True)].copy()
df_b2b['OrderDate'] = pd.to_datetime(df_b2b['OrderDate'])
snapshot_date = df_b2b['OrderDate'].max() + dt.timedelta(days=1)

rfm_b2b = df_b2b.groupby(['CustomerID', 'CompanyName', 'CurrentSegment']).agg({
    'OrderDate': lambda x: (snapshot_date - x.max()).days,  
    'OrderID': 'nunique',                                   
    'NetRevenue': 'sum',                                    
    'NetProfit': 'sum'                                      
}).reset_index()

rfm_b2b.rename(columns={'OrderDate': 'Recency', 'OrderID': 'Frequency', 'NetRevenue': 'Monetary'}, inplace=True)
rfm_b2b['CurrentSegment'] = rfm_b2b['CurrentSegment'].fillna('Unassigned B2B')

print("RFM Features successfully engineered.")

RFM Features successfully engineered.


### 3. Preprocessing, K-Means & Variance Analysis
Log transformations handle data skewness. After K-Means runs, we use a Random Forest Classifier to extract the feature importances (identifying which metrics drove the ML cluster definitions).

In [36]:
# 4. DATA PREPROCESSING, K-MEANS CLUSTERING & FEATURE IMPORTANCE
features = ['Recency', 'Frequency', 'Monetary', 'NetProfit']
rfm_log = rfm_b2b[features].copy()

# Log transformations
rfm_log['Recency'] = np.maximum(0, rfm_b2b['Recency'] - 210)
rfm_log['Recency'] = np.log(rfm_log['Recency'] + 1)
rfm_log['Frequency'] = np.log(rfm_log['Frequency'] + 1)
rfm_log['Monetary'] = np.log(rfm_log['Monetary'].clip(lower=1))
rfm_log['NetProfit'] = np.log(rfm_log['NetProfit'].clip(lower=1))

# Scaling & Clustering
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
rfm_b2b['ML_Cluster'] = kmeans.fit_predict(rfm_scaled)

# Variance Analysis (Feature Importance via RF)
rf_explainer = RandomForestClassifier(n_estimators=100, random_state=42)
rf_explainer.fit(rfm_scaled, rfm_b2b['ML_Cluster'])

print("\n--- FEATURE IMPORTANCES FOR K-MEANS CLUSTERING (Variance Analysis) ---")
for feat, imp in zip(features, rf_explainer.feature_importances_):
    print(f"{feat}: {imp*100:.2f}%")
print("-" * 65)


--- FEATURE IMPORTANCES FOR K-MEANS CLUSTERING (Variance Analysis) ---
Recency: 31.16%
Frequency: 23.50%
Monetary: 24.02%
NetProfit: 21.32%
-----------------------------------------------------------------


### 4. Dynamic Persona Mapping & Export
To solve the standard K-Means "Label Scrambling" issue, we dynamically assign personas based on actual centroid behaviors (Revenue and Recency). Finally, we print the summary matrix and export the dataset.

In [37]:
# 5. DYNAMIC ML PERSONA MAPPING
cluster_means = rfm_b2b.groupby('ML_Cluster')[['Recency', 'Monetary']].mean()

sorted_by_rev = cluster_means.sort_values('Monetary', ascending=False).index
top_2_clusters = sorted_by_rev[:2]
bottom_2_clusters = sorted_by_rev[2:]

# THE DISTINCTION BETWEEN THE TWO GROUPS WITH THE HIGHEST REVENUE.
if cluster_means.loc[top_2_clusters[0], 'Recency'] > cluster_means.loc[top_2_clusters[1], 'Recency']:
    churning_star, platinum = top_2_clusters[0], top_2_clusters[1] 
else:
    churning_star, platinum = top_2_clusters[1], top_2_clusters[0]
    
# THE DISTINCTION BETWEEN THE TWO GROUPS WITH THE LOWEST REVENUE.
if cluster_means.loc[bottom_2_clusters[0], 'Recency'] > cluster_means.loc[bottom_2_clusters[1], 'Recency']:
    sleeping_giant, active_regular = bottom_2_clusters[0], bottom_2_clusters[1] 
else:
    sleeping_giant, active_regular = bottom_2_clusters[1], bottom_2_clusters[0]

cluster_names = {
    platinum: 'Platinum Champions',
    churning_star: 'Churning Stars (High Risk)',
    sleeping_giant: 'Sleeping Giants',
    active_regular: 'Active Regulars (Low Margin)'
}

rfm_b2b.rename(columns={'Monetary': 'Revenue'}, inplace=True)
rfm_b2b['ML_Persona'] = rfm_b2b['ML_Cluster'].map(cluster_names)

# --- SUMMARY MATRIX ---
cluster_summary = rfm_b2b.groupby('ML_Persona').agg({
    'CustomerID': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Revenue': 'mean',
    'NetProfit': 'mean'
}).rename(columns={'CustomerID': 'Customer_Count'}).sort_values('Revenue', ascending=False)

print("\n--- B2B CLUSTER SUMMARY MATRIX ---")
display(cluster_summary.round(0).astype(int))

# --- EXPORT & PREVIEW ---
final_columns = [
    'CustomerID', 'CompanyName', 'CurrentSegment', 
    'ML_Cluster', 'ML_Persona', 'Recency', 'Frequency', 'Revenue', 'NetProfit'
]
df_b2b_ml_final = rfm_b2b[final_columns]

export_filename = 'RheinTrade_B2B_ML_Segments.csv'
df_b2b_ml_final.to_csv(export_filename, index=False)
print(f"\nSUCCESS: Segmentation exported to '{export_filename}'")

# Display the top 20 rows for final review
print("\n--- FINAL DATASET PREVIEW (TOP 20) ---")
display(df_b2b_ml_final.head(20))


--- B2B CLUSTER SUMMARY MATRIX ---


,Customer_Count,Recency,Frequency,Revenue,NetProfit
ML_Persona,,,,,
Platinum Champions,8,118,17,291439,126189
Churning Stars (High Risk),5,333,16,282837,124449
Sleeping Giants,3,177,5,207615,102463
Active Regulars (Low Margin),4,171,14,162241,70779



SUCCESS: Segmentation exported to 'RheinTrade_B2B_ML_Segments.csv'

--- FINAL DATASET PREVIEW (TOP 20) ---


,CustomerID,CompanyName,CurrentSegment,ML_Cluster,ML_Persona,Recency,Frequency,Revenue,NetProfit
0,1,Berlin Smart Hotel Group,B2B - Strategic Partner,2,Churning Stars (High Risk),244,20,368572.14,151535.79
1,2,München Tech Logistics,B2B - Strategic Partner,0,Active Regulars (Low Margin),208,18,195957.31,80716.00
2,3,Hamburg Maritime Solutions,B2B - Corporate,0,Active Regulars (Low Margin),172,13,126961.89,58372.74
3,4,Stuttgart Auto-Smart,B2B - Corporate,1,Platinum Champions,121,15,310488.65,131652.11
4,6,Frankfurt Data Center,B2B - Corporate,0,Active Regulars (Low Margin),157,12,146439.98,66014.50
5,19,Chicago Smart Office Inc.,B2B - Strategic Partner,1,Platinum Champions,1,22,436019.73,182194.70
6,20,NYC Secure Living,B2B - Strategic Partner,1,Platinum Champions,91,18,348051.01,147023.32
7,21,Austin Tech Rentals,B2B - Strategic Partner,0,Active Regulars (Low Margin),147,15,179604.79,78010.78
8,22,Silicon Valley IoT Lab,B2B - Corporate,1,Platinum Champions,96,14,226223.91,100794.24
9,31,Las Vegas Smart Casinoware,VIP Consumer,3,Sleeping Giants,269,4,155753.25,77167.55
